In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/tesla-directional-move-prediction-from-technical-indicators/sample_submission.csv
/kaggle/input/competitions/tesla-directional-move-prediction-from-technical-indicators/train.csv
/kaggle/input/competitions/tesla-directional-move-prediction-from-technical-indicators/test.csv


In [2]:
import os, pandas as pd

DATA_DIR = "/kaggle/input/competitions/tesla-directional-move-prediction-from-technical-indicators"

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(train_df.shape)
print(train_df.columns.tolist())
train_df.head()


(1200, 38)
['ID', 'y', 'SMA', 'EMA', 'ADX', 'Aroon', 'TDI-1', 'TDI-2', 'Donchian-1', 'Donchian-2', 'Donchian-3', 'VHF', 'CCI', 'MFI', 'OBV', 'MACD-1', 'MACD-2', 'RSI', 'Stochastics-1', 'Stochastics-2', 'Stochastics-3', 'CMO', 'KST-1', 'KST-2', 'TRIX-1', 'TRIX-2', 'ROC', 'DVI-1', 'DVI-2', 'DVI-3', 'Momentum', 'Bbands-1', 'Bbands-2', 'Bbands-3', 'Volatility', 'Pbands-1', 'Pbands-2', 'Pbands-3']


,ID,y,SMA,EMA,ADX,Aroon,TDI-1,TDI-2,Donchian-1,Donchian-2,...,DVI-2,DVI-3,Momentum,Bbands-1,Bbands-2,Bbands-3,Volatility,Pbands-1,Pbands-2,Pbands-3
0,1,up,305.154164,309.161711,9.693882,-40,-842.976654,-839.650085,299.626679,307.775009,...,0.533333,0.613333,20.809998,277.927543,332.380786,0.542651,0.529606,276.421546,305.154164,333.886782
1,2,up,21.284367,20.954702,13.942328,-25,-54.582664,-63.203333,19.578667,20.326000,...,0.633333,0.820000,0.391333,18.454787,24.113946,0.424895,0.338541,19.030516,21.284367,23.538218
2,3,up,295.992830,307.554733,23.729694,50,290.966644,-1052.096649,268.523346,317.241669,...,1.000000,0.973333,25.973328,238.151152,353.834507,1.087412,0.466613,247.569375,295.992830,344.416285
3,4,neutral,242.628500,244.038023,25.278022,65,-133.479965,517.899994,244.666672,249.741669,...,0.900000,0.420000,4.166672,226.495372,258.761629,0.800670,0.220938,232.715056,242.628500,252.541944
4,5,up,254.895333,261.501230,21.167000,100,-2606.700119,174.953354,247.139999,279.508339,...,1.000000,0.866667,16.376678,213.337363,296.453304,1.009906,0.661151,228.954502,254.895333,280.836164


In [3]:
# ====================================================
# 0. Kaggle template imports
# ====================================================
import numpy as np
import pandas as pd
import os

# Inspect input directory (optional)
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# ====================================================
# 1. Additional imports (PyTorch, sklearn, etc.)
# ====================================================
import time
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score


/kaggle/input/competitions/tesla-directional-move-prediction-from-technical-indicators/sample_submission.csv
/kaggle/input/competitions/tesla-directional-move-prediction-from-technical-indicators/train.csv
/kaggle/input/competitions/tesla-directional-move-prediction-from-technical-indicators/test.csv


In [4]:
# ====================================================
# 2. Reproducibility and device setup
# ====================================================
def set_seed(seed: int = 42) -> None:
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


In [5]:
# ====================================================
# 3. Data loading for Tesla Directional Move Prediction
# ====================================================
DATA_DIR = "/kaggle/input/competitions/tesla-directional-move-prediction-from-technical-indicators"

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print(train_df.head())
print(train_df.columns.tolist())


Train shape: (1200, 38)
Test shape: (532, 37)
   ID        y         SMA         EMA        ADX  Aroon        TDI-1  \
0   1       up  305.154164  309.161711   9.693882    -40  -842.976654   
1   2       up   21.284367   20.954702  13.942328    -25   -54.582664   
2   3       up  295.992830  307.554733  23.729694     50   290.966644   
3   4  neutral  242.628500  244.038023  25.278022     65  -133.479965   
4   5       up  254.895333  261.501230  21.167000    100 -2606.700119   

         TDI-2  Donchian-1  Donchian-2  ...     DVI-2     DVI-3   Momentum  \
0  -839.650085  299.626679  307.775009  ...  0.533333  0.613333  20.809998   
1   -63.203333   19.578667   20.326000  ...  0.633333  0.820000   0.391333   
2 -1052.096649  268.523346  317.241669  ...  1.000000  0.973333  25.973328   
3   517.899994  244.666672  249.741669  ...  0.900000  0.420000   4.166672   
4   174.953354  247.139999  279.508339  ...  1.000000  0.866667  16.376678   

     Bbands-1    Bbands-2  Bbands-3  Volatilit

In [6]:
# ====================================================
# 4. Column configuration and feature selection
# ====================================================
# From the provided columns:
# ['ID', 'y', 'SMA', 'EMA', 'ADX', 'Aroon', 'TDI-1', 'TDI-2', 'Donchian-1', 'Donchian-2',
#  'Donchian-3', 'VHF', 'CCI', 'MFI', 'OBV', 'MACD-1', 'MACD-2', 'RSI',
#  'Stochastics-1', 'Stochastics-2', 'Stochastics-3', 'CMO', 'KST-1', 'KST-2',
#  'TRIX-1', 'TRIX-2', 'ROC', 'DVI-1', 'DVI-2', 'DVI-3', 'Momentum',
#  'Bbands-1', 'Bbands-2', 'Bbands-3', 'Volatility', 'Pbands-1', 'Pbands-2', 'Pbands-3']

# We treat this as a single time series ordered by ID (no separate stock/date columns).
ID_COL = "ID"
TARGET_COL = "y"

# Sort by ID to ensure correct temporal order
train_df = train_df.sort_values(ID_COL).reset_index(drop=True)

# Define feature columns: drop ID and y, use all technical indicators as features
exclude = {ID_COL, TARGET_COL}
feature_cols = [c for c in train_df.columns if c not in exclude]

print("Number of feature columns:", len(feature_cols))
print("First few features:", feature_cols[:5])


Number of feature columns: 36
First few features: ['SMA', 'EMA', 'ADX', 'Aroon', 'TDI-1']


In [7]:
# ====================================================
# 5. Create lagged features for a single time series
# ====================================================
def create_lagged_features_single_series(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    id_col: str,
    window: int = 10
) -> tuple[np.ndarray, np.ndarray]:
    """Create lagged feature matrix and target vector from a single ordered time series.

    We assume df is already sorted by id_col, which can be interpreted as time index.
    For each position t, input is the previous `window` steps' features,
    and label is based on the next step (t+1) target.
    """
    df = df.sort_values(id_col).reset_index(drop=True)
    feats = df[feature_cols].values
    targets_raw = df[target_col].values

    # Convert 'y' from string labels to binary (1: up, 0: otherwise)
    # You can refine this mapping depending on the competition definition.
    # Example: treat 'up' as 1, others ('down', 'neutral') as 0.
    targets = np.array([1 if v == "up" else 0 for v in targets_raw], dtype=np.int64)

    X_list: list[np.ndarray] = []
    y_list: list[int] = []

    # Use a simple univariate time series view (no grouping)
    for idx in range(window, len(df) - 1):
        window_feats = feats[idx - window:idx, :]  # (window, num_features)
        X_list.append(window_feats.flatten())

        y_label = targets[idx + 1]  # label for the next step
        y_list.append(y_label)

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=np.int64)
    return X, y


WINDOW = 10
X_all, y_all = create_lagged_features_single_series(
    train_df,
    feature_cols,
    TARGET_COL,
    ID_COL,
    window=WINDOW
)

print("Lagged features shape:", X_all.shape)
print("Labels shape:", y_all.shape)
print("Positive class ratio:", y_all.mean())


Lagged features shape: (1189, 360)
Labels shape: (1189,)
Positive class ratio: 0.37342304457527337


In [8]:
# ====================================================
# 6. Time-based train/validation split using ID as time index
# ====================================================
min_id = train_df[ID_COL].min()
max_id = train_df[ID_COL].max()
print("ID range:", min_id, "to", max_id)

split_point = min_id + int(0.8 * (max_id - min_id))
print("Split point:", split_point)

train_mask = train_df[ID_COL] <= split_point
val_mask = train_df[ID_COL] > split_point

train_df_ts = train_df[train_mask].copy()
val_df_ts = train_df[val_mask].copy()

X_train, y_train = create_lagged_features_single_series(
    train_df_ts,
    feature_cols,
    TARGET_COL,
    ID_COL,
    window=WINDOW
)
X_val, y_val = create_lagged_features_single_series(
    val_df_ts,
    feature_cols,
    TARGET_COL,
    ID_COL,
    window=WINDOW
)

print("Time-based train shape:", X_train.shape, y_train.shape)
print("Time-based val shape:", X_val.shape, y_val.shape)


ID range: 1 to 1200
Split point: 960
Time-based train shape: (949, 360) (949,)
Time-based val shape: (229, 360) (229,)


In [9]:
# ====================================================
# 7. Scaling (fit on train, apply to val)
# ====================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled val shape:", X_val_scaled.shape)


Scaled train shape: (949, 360)
Scaled val shape: (229, 360)


In [10]:
# ====================================================
# 8. PyTorch Dataset & DataLoader for tabular (MLP)
# ====================================================
class TabularDataset(Dataset):
    """Dataset for tabular (flattened) features and binary labels."""

    def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()

    def __len__(self) -> int:
        return self.X.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]


BATCH_SIZE = 256

train_ds = TabularDataset(X_train_scaled, y_train)
val_ds = TabularDataset(X_val_scaled, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)


In [11]:
# ====================================================
# 9. Baseline MLP model
# ====================================================
class MLPBaseline(nn.Module):
    """Baseline MLP model for lagged tabular features."""

    def __init__(self, input_dim: int, hidden_dims: list[int] | None = None) -> None:
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [128, 64]

        layers: list[nn.Module] = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass producing logits of shape (batch,)."""
        logits = self.net(x).squeeze(-1)
        return logits


In [12]:
# ====================================================
# 10. Training / evaluation utilities (shared)
# ====================================================
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module
) -> float:
    """Train model for one epoch and return average loss."""
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb.float())
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module
) -> tuple[float, float, float]:
    """Evaluate model and return (avg_loss, accuracy, auc)."""
    model.eval()
    total_loss = 0.0
    all_logits: list[torch.Tensor] = []
    all_targets: list[torch.Tensor] = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb.float())

            total_loss += loss.item() * xb.size(0)
            all_logits.append(logits.detach().cpu())
            all_targets.append(yb.detach().cpu())

    all_logits_t = torch.cat(all_logits)
    all_targets_t = torch.cat(all_targets)

    probs = torch.sigmoid(all_logits_t).numpy()
    preds = (probs > 0.5).astype(int)
    targets_np = all_targets_t.numpy()

    acc = accuracy_score(targets_np, preds)
    try:
        auc = roc_auc_score(targets_np, probs)
    except ValueError:
        auc = float("nan")

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, acc, auc


In [13]:
# ====================================================
# 11. Train the MLP baseline
# ====================================================
input_dim = X_train_scaled.shape[1]
mlp_model = MLPBaseline(input_dim=input_dim).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-3)

NUM_EPOCHS = 300

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(mlp_model, train_loader, optimizer, criterion)
    val_loss, val_acc, val_auc = evaluate(mlp_model, val_loader, criterion)
    dt = time.time() - t0

    print(
        f"[MLP] Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f} | "
        f"val_auc={val_auc:.4f} | "
        f"time={dt:.1f}s"
    )


[MLP] Epoch 01 | train_loss=0.6768 | val_loss=0.6628 | val_acc=0.6201 | val_auc=0.5513 | time=0.2s
[MLP] Epoch 02 | train_loss=0.6430 | val_loss=0.6568 | val_acc=0.6201 | val_auc=0.5747 | time=0.0s
[MLP] Epoch 03 | train_loss=0.6270 | val_loss=0.6575 | val_acc=0.6201 | val_auc=0.5789 | time=0.0s
[MLP] Epoch 04 | train_loss=0.6093 | val_loss=0.6571 | val_acc=0.6288 | val_auc=0.5814 | time=0.0s
[MLP] Epoch 05 | train_loss=0.5910 | val_loss=0.6569 | val_acc=0.6376 | val_auc=0.5775 | time=0.0s
[MLP] Epoch 06 | train_loss=0.5715 | val_loss=0.6596 | val_acc=0.6288 | val_auc=0.5785 | time=0.0s
[MLP] Epoch 07 | train_loss=0.5510 | val_loss=0.6654 | val_acc=0.6026 | val_auc=0.5763 | time=0.0s
[MLP] Epoch 08 | train_loss=0.5258 | val_loss=0.6775 | val_acc=0.6026 | val_auc=0.5750 | time=0.0s
[MLP] Epoch 09 | train_loss=0.4972 | val_loss=0.6864 | val_acc=0.5808 | val_auc=0.5733 | time=0.0s
[MLP] Epoch 10 | train_loss=0.4656 | val_loss=0.6991 | val_acc=0.5808 | val_auc=0.5675 | time=0.0s


In [21]:
# ====================================================
# Submission generation with MLP baseline
# ====================================================

WINDOW = 10

test_df = test_df.sort_values("ID").reset_index(drop=True)
X_test_feats = test_df[feature_cols].values
N_test = len(test_df)

# 1) lagged features for indices [WINDOW, ..., N_test-1]
X_test_lag = create_lagged_features_single_series_test(X_test_feats, window=WINDOW)
test_ids_aligned = test_df["ID"].values[WINDOW:]  # length 522

# 2) scale and predict
X_test_lag_scaled = scaler.transform(X_test_lag)

mlp_model.eval()
with torch.no_grad():
    X_test_tensor = torch.from_numpy(X_test_lag_scaled).float().to(device)
    logits = mlp_model(X_test_tensor)
    probs = torch.sigmoid(logits).cpu().numpy()

pred_binary = (probs > 0.5).astype(int)
pred_labels = np.where(pred_binary == 1, "up", "neutral")

# 3) build full submission with 532 rows
full_ids = test_df["ID"].values  # length 532

# default: all "neutral"
full_labels = np.array(["neutral"] * N_test, dtype=object)

# fill from WINDOW onward with model predictions
full_labels[WINDOW:] = pred_labels

submission = pd.DataFrame({
    "ID": full_ids,
    "y": full_labels
})

print(submission.shape)  # (532, 2)
submission.to_csv("mlp_baseline_submission.csv", index=False)

(532, 2)


In [14]:
# ====================================================
# 12. Helper: reshape flattened X into sequences for RNNs
# ====================================================
def make_sequence_dataset(
    X_flat: np.ndarray,
    y: np.ndarray,
    window: int,
    num_features: int
) -> tuple[np.ndarray, np.ndarray]:
    """Reshape flattened lag features into (N, T, F) sequences."""
    N, D = X_flat.shape
    assert D == window * num_features, "Inconsistent dimensions for window and num_features."
    X_seq = X_flat.reshape(N, window, num_features)
    return X_seq, y


num_features = len(feature_cols)
X_train_seq, y_train_seq = make_sequence_dataset(X_train_scaled, y_train, WINDOW, num_features)
X_val_seq, y_val_seq = make_sequence_dataset(X_val_scaled, y_val, WINDOW, num_features)

print("Sequence train shape:", X_train_seq.shape, y_train_seq.shape)
print("Sequence val shape:", X_val_seq.shape, y_val_seq.shape)


Sequence train shape: (949, 10, 36) (949,)
Sequence val shape: (229, 10, 36) (229,)


In [15]:
# ====================================================
# 13. Sequence Dataset & DataLoader for RNNs
# ====================================================
class SequenceDataset(Dataset):
    """Dataset for sequence data (N, T, F) and binary labels."""

    def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()

    def __len__(self) -> int:
        return self.X.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]


train_seq_ds = SequenceDataset(X_train_seq, y_train_seq)
val_seq_ds = SequenceDataset(X_val_seq, y_val_seq)

train_seq_loader = DataLoader(train_seq_ds, batch_size=BATCH_SIZE, shuffle=True)
val_seq_loader = DataLoader(val_seq_ds, batch_size=BATCH_SIZE, shuffle=False)


In [16]:
# ====================================================
# 14. GRU-based classifier (TODO for students)
# ====================================================
class GRUClassifier(nn.Module):
    """GRU-based sequence classifier."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        num_layers: int = 1
    ) -> None:
        super().__init__()
        # TODO: define self.gru and self.fc
        # Example:
        # self.gru = nn.GRU(input_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        # self.fc = nn.Linear(hidden_dim, 1)
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: x shape (batch, T, input_dim), return logits (batch,)."""
        # TODO: implement forward using last hidden state
        # Example:
        # out, h_n = self.gru(x)
        # last_h = h_n[-1]  # (batch, hidden_dim)
        # logits = self.fc(last_h).squeeze(-1)
        raise NotImplementedError


In [17]:
# ====================================================
# 15. minGRU cell and classifier (TODO for students)
#     Hint (minimal GRU-style update):
#       z_t = sigmoid(W_z x_t + U_z h_{t-1} + b_z)
#       h_t_candidate = tanh(W_h x_t + U_h h_{t-1} + b_h)
#       h_t = z_t * h_{t-1} + (1 - z_t) * h_t_candidate
# ====================================================
class MinGRUCell(nn.Module):
    """A minimal GRU-like recurrent cell."""

    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        # TODO: define parameters for the minimal GRU cell
        # Example:
        # self.W_z = nn.Linear(input_dim, hidden_dim)
        # self.U_z = nn.Linear(hidden_dim, hidden_dim, bias=False)
        # self.W_h = nn.Linear(input_dim, hidden_dim)
        # self.U_h = nn.Linear(hidden_dim, hidden_dim, bias=False)
        raise NotImplementedError

    def forward(
        self,
        x_t: torch.Tensor,
        h_prev: torch.Tensor
    ) -> torch.Tensor:
        """Compute next hidden state h_t from x_t and h_prev."""
        # TODO: implement the minGRU update rule
        # Example:
        # z_t = torch.sigmoid(self.W_z(x_t) + self.U_z(h_prev))
        # h_t_candidate = torch.tanh(self.W_h(x_t) + self.U_h(h_prev))
        # h_t = z_t * h_prev + (1 - z_t) * h_t_candidate
        raise NotImplementedError


class MinGRUClassifier(nn.Module):
    """Sequence classifier built on top of a MinGRUCell."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64
    ) -> None:
        super().__init__()
        # TODO: create a MinGRUCell instance and a final linear layer
        # Example:
        # self.cell = MinGRUCell(input_dim, hidden_dim)
        # self.fc = nn.Linear(hidden_dim, 1)
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass for minGRU classifier: x shape (batch, T, input_dim)."""
        # TODO: unroll the cell over time and use last hidden state
        # Example:
        # batch_size, T, F = x.shape
        # h = torch.zeros(batch_size, hidden_dim, device=x.device)
        # for t in range(T):
        #     x_t = x[:, t, :]
        #     h = self.cell(x_t, h)
        # logits = self.fc(h).squeeze(-1)
        raise NotImplementedError


In [18]:
# ====================================================
# 16. Generic training loop for RNN models
# ====================================================
def train_rnn_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = 10,
    lr: float = 1e-3
) -> dict[str, list[float]]:
    """Generic training loop for sequence models."""
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    history: dict[str, list[float]] = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
        "val_auc": [],
        "epoch_time": [],
    }

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, val_auc = evaluate(model, val_loader, criterion)
        dt = time.time() - t0

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_auc"].append(val_auc)
        history["epoch_time"].append(dt)

        print(
            f"[RNN] Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f} | "
            f"val_auc={val_auc:.4f} | "
            f"time={dt:.1f}s"
        )

    return history


In [19]:
# ====================================================
# 17. Example usage (uncomment after implementing TODOs)
# ====================================================
# gru_model = GRUClassifier(input_dim=num_features, hidden_dim=64, num_layers=1)
# gru_history = train_rnn_model(gru_model, train_seq_loader, val_seq_loader, num_epochs=10, lr=1e-3)
#
# mingru_model = MinGRUClassifier(input_dim=num_features, hidden_dim=64)
# mingru_history = train_rnn_model(mingru_model, train_seq_loader, val_seq_loader, num_epochs=10, lr=1e-3)
